<a href="https://colab.research.google.com/github/BBVA/mercury-graph/blob/master/tutorials/mercury-graph-tutorial-banksim-nospark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Table of contents
- [What is `mercury-graph`?](#mercury-graph)
- [Environment setup](#environment-setup)
- [Graph creation](#graph-creation)
- [Graph embeddings](#graph-embeddings)
- [Spectral clustering](#spectral)
- [Transition matrix (Markov chains)](#transition)

# What is `mercury-graph`? <a name="mercury-graph"></a>

**`mercury-graph`** is a Python library that offers **graph analytics capabilities with a technology-agnostic API**, enabling users to apply a curated range of performant and scalable algorithms and utilities regardless of the underlying data framework. The consistent, scikit-like interface abstracts away the complexities of internal transformations, allowing users to effortlessly switch between different graph representations to leverage optimized algorithms implemented using pure Python, [**numba**](https://numba.pydata.org/), [**networkx**](https://networkx.org/) and PySpark [**GraphFrames**](https://graphframes.github.io/graphframes/docs/_site/index.html).

It is a part of [**`mercury`**](https://www.bbvaaifactory.com/mercury/), a collaborative library developed by the **Advanced Analytics community at BBVA** that offers a broad range of tools to simplify and accelerate data science workflows. This library was originally an Inner Source project, but some components, like `mercury.graph`, have been released as Open Source.

Currently implemented **submodules** in `mercury.graph` include:
- [**`mercury.graph.core`**](#graph-creation), with the main classes of the library that create and store the graphs' data and properties.
- **`mercury.graph.ml`**, with graph theory and machine learning algorithms such as [Louvain community detection](#louvain), [spectral clustering](#spectral), [Markov chains](#transition), [spreading activation-based diffusion models](#spread-activation) and graph random walkers.
- **`mercury.graph.embeddings`**, with classes that calculate [graph embeddings](#graph-embeddings) in different ways, such as following the [Node2Vec](#node2vec) algorithm.


# Environment setup <a name="environment-setup"></a>



<div class="alert alert-block alert-info">
<b>Note:</b> This notebook will showcase methods in `mercury.graph` that do not require configuration of a Spark cluster.
</div>

In [ ]:
# Mercury-Graph
! pip install mercury-graph

From `mercury.graph`, we first **import `Graph`**, to create graphs **from pandas/Spark dataframes or from networkx/graphframes graph objects**. It is the core class of the library, storing the graphs' data and properties and offering a **flexible and technology-agnostic API**.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', None)

import networkx as nx

from mercury.graph.core import Graph

import matplotlib.pyplot as plt
import seaborn as sns

# Graph creation <a name="graph-creation"></a>

We will create a graph `g` based on the [**BankSim dataset**](https://www.researchgate.net/publication/265736405_BankSim_A_Bank_Payment_Simulation_for_Fraud_Detection_Research), which contains **synthetic transactional data**. Each row of the dataset represents a generated transaction from a customer to a merchant, and a flag that indicates if the transaction has been detected as fraudulent. Customer IDs begin with 'C' and merchants IDs with 'M'.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/atavci/fraud-detection-on-banksim-data/refs/heads/master/Data/synthetic-data-from-a-financial-payment-system/bs140513_032310.csv",
                 quotechar ="'")

print(f"Number of rows: {len(df)}")
df.head()

Let's obtain **which nodes are involved in a transaction that was marked as fraud**:

In [ ]:
fraud_ids = df[df["fraud"] == 1]["customer"].unique().tolist()
print(f"Number of customers involved in a transaction marked as fraud: {len(fraud_ids)}")

For each transaction, only the customer, merchant, amount and fraud columns will be kept and considered to build the nodes and edges of the graph. Given that there may be several transactions of different categories between two nodes (customer and merchant), these **transactions will be aggregated to keep only one edge between each pair of nodes**.

The dataset as-is represents the interaction flows from customers to merchants. In this tutorial, we will also create the **reversed edges**, from merchants to customers. In this way, when using algorithms that depend on random walks, paths from merchants to customers will also be allowed.

In [ ]:
df_edges = df[["customer", "merchant", "amount"]] \
              .groupby(["customer", "merchant"]) \
              .agg({"amount": "sum"}) \
              .reset_index()

df_edges_reversed = pd.DataFrame()
df_edges_reversed["customer"] = df_edges["merchant"].values
df_edges_reversed["merchant"] = df_edges["customer"].values
df_edges_reversed["amount"] = df_edges["amount"].values

df_edges = pd.concat([df_edges, df_edges_reversed])
del df_edges_reversed

print(f"Number of edges: {len(df_edges)}")

Fraud will be considered as a node attribute, highlighting customers that were involved in a fraudulent transaction.

In [ ]:
df_nodes = pd.DataFrame(df_edges["customer"].unique(), columns=["node_id"])
df_nodes["fraud"] = 0
df_nodes.loc[df_nodes["node_id"].isin(fraud_ids), "fraud"] = 1

print(f"Number of nodes: {len(df_nodes)}")

In [ ]:
g = Graph(data=df_edges,
          nodes=df_nodes,
          keys={"src": "customer",
                "dst": "merchant",
                "weight": "amount",
                "id": "node_id"})

print(g)

# Graph embeddings <a name="graph-embeddings"></a>

Class `GraphEmbeddings` of mercury.graph.embeddings creates an **embedding mapping the nodes of a graph by doing random walks**, implemented using numpy and networkx. These walks start from a random node and select the edges with a probability that is proportional to the **weight** of the edge.

In [ ]:
from mercury.graph.embeddings import GraphEmbedding

In [ ]:
ge = GraphEmbedding(dimension=100,
                    n_jumps=500000,
                    max_per_epoch=50000,
                    learn_step=2,
                    bidirectional=True)

print(ge)

In [ ]:
ge.fit(g)

After fitting the object to the subgraph, an `Embedding` object is created, containing the representation of the vector embeddings matrix.

In [ ]:
print(ge.embedding(), "\n")

ge_em_np = ge.embedding().as_numpy()
print(f"Shape: {ge_em_np.shape} \n")
print(ge_em_np)

For each node, the **most similar nodes and the similarity metric** (by default, cosine similarity) can be obtained using method `get_most_similar_nodes`. Essentially, it fetches the most similar embeddings using the underlying `Embedding` object.

Let's get the most similar nodes to a node (customer) that was involved in two fraudulent transactions:

In [ ]:
customer_id = "C967956630"

df[(df["customer"]==customer_id) & (df["fraud"]==1)]

In [ ]:
similar_embeddings = ge.get_most_similar_nodes(customer_id, 10)
similar_embeddings

We can observe that many of the most similar customers were also involved in fraudulent transactions.

In [ ]:
df[df["customer"].isin(similar_embeddings["word"].tolist())].groupby("customer").agg(fraud=("fraud", lambda x: x.sum() >= 1))

Now we can obtain the embeddings of the nodes. Let's create a dataframe with the embeddings and also mark which nodes were in a transaction that was marked as fraud:

In [ ]:
ge_em_df = pd.DataFrame({"vector": [v for v in ge_em_np]})

ge_em_df["customer"] = list(g.networkx.nodes)
ge_em_df["fraud"] = 0
ge_em_df.loc[ge_em_df["customer"].isin(fraud_ids), "fraud"] = 1

ge_em_df.head(1)

[TSNE](https://scikit-learn.org/1.5/modules/generated/sklearn.manifold.TSNE.html) can be used to reduce the embedding dimension to two, making it easier to create visualizations. Let's visualize the node embeddings, using the color to see which nodes were involved in transactions marked as fraudulent. We can see that, in general, a lot of those nodes are located in a separate area of the embedding space:

In [ ]:
from sklearn.manifold import TSNE

ge_tsne = TSNE(perplexity=12.0, metric='euclidean', random_state=1)
ge_tsne_np = ge_tsne.fit_transform(np.stack(ge_em_df["vector"].values))

ge_tsne_pd = pd.DataFrame(ge_tsne_np)
ge_tsne_pd["fraud"] = ge_em_df["fraud"].values
ge_tsne_pd["id"] = ge_em_df["customer"].values

fig, axes = plt.subplots(1, 1, figsize=(12, 6))
ax = sns.scatterplot(x=0, y=1,
                     hue="fraud", palette={0: "green", 1: "red"},
                     data=ge_tsne_pd)

# Spectral clustering <a name="spectral"></a>

Class `SpectralClustering` from `mercury.graph.ml` implements the **unsupervised [spectral clustering algorithm](https://www.sciencedirect.com/topics/computer-science/spectral-clustering)** to group nodes in a graph. This algorithm can work in two modes: "networkx" (running the algorithm locally, with a methodology similar to [scikit-learn's implementation](https://scikit-learn.org/1.5/modules/generated/sklearn.cluster.SpectralClustering.html) but expecting a graph object instead of a numpy array) or "spark" (using PySpark and graphframes).

In [ ]:
from mercury.graph.ml import SpectralClustering

In [ ]:
sc = SpectralClustering(n_clusters=4, mode="networkx")

print(sc)

As with `LouvainCommunities`, cluster assignments are available after fitting through the pandas dataframe `.labels_`, following the `scikit-learn` convention:

In [ ]:
sc.fit(g)

In [ ]:
sc_df = sc.labels_.rename(columns={"node_id": "id"})

sc_df

Again, given the size and complexity of the graph, a viable option for visualizing the detected clusters is to color the embedding space created by [`GraphEmbedding`](#graph-embeddings) and reduced to two dimensions by using [TSNE](https://scikit-learn.org/1.5/modules/generated/sklearn.manifold.TSNE.html). This approach allows us to verify that the clusters created make sense, although it does not show information on the edges or the structure of the graph.

In [ ]:
sc_ge_tsne_df = ge_tsne_pd.merge(sc_df, on = "id", how = "inner", validate = None)
sc_ge_tsne_df["cluster"] = sc_ge_tsne_df["cluster"].astype(str)

fig, axes = plt.subplots(1, 1, figsize=(12, 6))
ax = sns.scatterplot(x=0, y=1, hue="cluster", data=sc_ge_tsne_df)

[**Modularity**](https://en.wikipedia.org/wiki/Modularity_(networks)) is a metric that measures the strength of the division of a graph into clusters:

In [ ]:
print(f"Modularity: {sc.modularity_}")

Spectral clustering can also be performed using PySpark under the hood by simply passing `mode="spark"` to the constructor, which uses the `Graph` object's graphframe property.

# Transition matrix (Markov chains) <a name="transition"></a>

Class `Transition` of `mercury.graph.ml` can be used to obtain the **transition matrix** of a graph, which computes the distribution of probability of being in each of the nodes (or states) of a directed graph (or Markov process).

In [ ]:
from mercury.graph.ml import Transition

In [ ]:
tm = Transition().fit(g).to_pandas()

tm